In [7]:
import sqlite3
import pandas as pd
import duckdb
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from hdbscan import HDBSCAN
from huggingface_hub import notebook_login
from umap import UMAP
from bertopic.vectorizers import ClassTfidfTransformer

notebook_login()

In [3]:

# 1. Load the local CSV into a Pandas DataFrame
df = pd.read_csv("bronze/games.csv")

# 2. Create an in-memory SQLite database connection
conn = sqlite3.connect(":memory:")

# 3. Write the DataFrame to the SQL database as a table
df.to_sql("games", conn, index=False, if_exists="replace")

# 4. Run your SQL query
query = "SELECT * FROM games LIMIT 10"
result_df = pd.read_sql_query(query, conn)

print(result_df)

# 5. Clean up connection
conn.close()


                                   AppID          Name Release date  \
0             Black Dragon Mage Playtest   Aug 1, 2023        0 - 0   
1  Supipara - Chapter 1 Spring Has Come!  Jul 29, 2016    0 - 20000   
2      Mystery Solitaire The Black Raven   May 6, 2019    0 - 20000   
3            버튜버 파라노이아 - Vtuber Paranoia  Oct 31, 2024    0 - 20000   
4                          Maze Quest VR  Apr 24, 2025    0 - 20000   
5                               Agony VR   Apr 5, 2023    0 - 20000   
6                     Armored Brigade II   Apr 8, 2025    0 - 20000   
7         Galacatraz: Eject Equip Escape   Feb 1, 2018    0 - 20000   
8                            Hepta Beats   May 7, 2021    0 - 20000   
9                 MUMBA IV: Egypt Jewels  Dec 13, 2021    0 - 20000   

   Estimated owners  Peak CCU  Required age  Price  DiscountDLC count  \
0                 0         0          0.00      0                  0   
1                 0         0          5.24     65                  0   

In [14]:


# DuckDB can run the regex query directly on the CSV file 
# and write the results to a new CSV file in a single line:
query = """
        SELECT appid 
        FROM 'bronze/extended_steam_games.csv' 
        WHERE positive IS NULL 
           OR negative IS NULL
           OR owners IS NULL
           OR average_forever IS NULL
           OR developer IS NULL
           OR publisher IS NULL
        LIMIT 20
"""

duckdb.sql(query).show()


┌─────────┐
│  appid  │
│  int64  │
├─────────┤
│   40800 │
│   12900 │
│  247120 │
│ 2713550 │
│ 1789480 │
│ 1108320 │
│  385770 │
│  343780 │
│  396160 │
│  353380 │
│  214190 │
│  258180 │
│ 1110390 │
│  253940 │
│   26500 │
│  695920 │
│  208730 │
│  314450 │
│  247910 │
│  253650 │
└─────────┘
  20 rows



In [35]:
query = """
       SELECT *  
       FROM 'bronze/extended_steam_games.csv' 
       WHERE supported_languages ILIKE '%english%'
       LIMIT 5;

"""

duckdb.sql(query).show(max_width=10000)

┌─────────┬──────────────────────────────────┬──────────────────┬─────────────────┬──────────┬──────────┬────────────────────────────┬─────────────────┬────────────────┬────────────────┬───────────────┬───────┬──────────────┬──────────┬─────────┬──────────────┬──────────────┬─────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────

In [ ]:
# send the cleaned files with price tier to a file

query = """
    COPY (
        SELECT appid, owners, name, positive, negative, publisher, developer, average_forever, categories, short_description,
        positive / (positive + negative) AS percent_positive,
        positive / (positive + negative) AS percent_negative,
        positive + negative AS total,
        CASE 
            WHEN price = 0 THEN 'Free'
            WHEN price > 0 AND price <= 20 THEN 'Indie'
            WHEN price > 20 AND price <= 40 THEN 'A'
            WHEN price > 40 AND price <= 70 THEN 'AA'
            ELSE 'AAA'
        END AS price_tier,
        FROM 'bronze/extended_steam_games.csv' 
        WHERE NOT (positive IS NULL 
                   OR negative IS NULL
                   OR owners IS NULL
                   OR average_forever IS NULL)
                   AND supported_languages ILIKE '%english%'
            

    ) TO 'silver/cleaned_extended_steam_game_price_tiers.csv' (HEADER, DELIMITER ',');
"""

duckdb.sql(query)

In [36]:
# send potentially malformed records to another file

query = """
    COPY (
        SELECT appid, owners, name, positive, negative, publisher, developer, average_forever, categories, short_description,
        positive / (positive + negative) AS percent_positive,
        positive / (positive + negative) AS percent_negative,
        positive + negative AS total,
        CASE 
            WHEN price = 0 THEN 'Free'
            WHEN price > 0 AND price <= 20 THEN 'Indie'
            WHEN price > 20 AND price <= 40 THEN 'A'
            WHEN price > 40 AND price <= 70 THEN 'AA'
            ELSE 'AAA'
        END AS price_tier,
        FROM 'bronze/extended_steam_games.csv' 
        WHERE (positive IS NULL 
                OR negative IS NULL
                OR owners IS NULL
                OR average_forever IS NULL)
                AND supported_languages ILIKE '%english%'

    ) TO 'silver/malformed_extended_steam_game_price_tiers.csv' (HEADER, DELIMITER ',');
"""

duckdb.sql(query)

In [18]:
query = """
        SELECT price_tier, SUM(total) AS total_votes, SUM(positive) AS positive, SUM(negative) AS negative,
        ROUND(SUM(positive) / SUM(total), 2) AS percent_positive, ROUND(SUM(negative) / SUM(total), 2) AS percent_negative
        FROM 'silver/cleaned_extended_steam_game_price_tiers.csv'
        GROUP BY price_tier
        ORDER BY percent_positive DESC
"""

duckdb.sql(query).show()

┌────────────┬─────────────┬──────────┬──────────┬──────────────────┬──────────────────┐
│ price_tier │ total_votes │ positive │ negative │ percent_positive │ percent_negative │
│  varchar   │   int128    │  int128  │  int128  │      double      │      double      │
├────────────┼─────────────┼──────────┼──────────┼──────────────────┼──────────────────┤
│ AAA        │   108785508 │ 95680180 │ 13105328 │             0.88 │             0.12 │
│ AA         │      188360 │   161002 │    27358 │             0.85 │             0.15 │
│ A          │        3213 │     2714 │      499 │             0.84 │             0.16 │
│ Free       │    36079997 │ 28919566 │  7160431 │              0.8 │              0.2 │
└────────────┴─────────────┴──────────┴──────────┴──────────────────┴──────────────────┘



Cleaning games.csv

In [ ]:
# explore games.csv to see what we have

query = """
       SELECT *  FROM 'bronze/games.csv' LIMIT 5;

"""

duckdb.sql(query).show(max_width=10000)

┌──────────┬───────────────────────────────────────┬──────────────┬───────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
query = """
       SELECT *  
       FROM 'bronze/weighted_score_above_08.csv'
       WHERE language = 'english'
       LIMIT 5;

"""

duckdb.sql(query).show()

┌──────────────────┬───────┬────────────────┬───────────────────┬────────────────────────┬────────────────────┬─────────────────────────┬────────────────────────────────┬───────────────────────────┬────────────────────┬──────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────┬───────────────────┬──────────┬──────────┬─────────────┬─────────────────────┬───────────────┬────────────────┬───────────────────┬─────────────────────────────┬───────────────────────┬──────────────────────┐
│ recommendationid │ appid │      game      │  author_steamid   │ author_num_games_owned │ author_num_reviews │ author_playtime_forever │ author_playtime_last_two_weeks │ author_playtime_at_review │ author_last_played │ language │                                                                                             

In [39]:
query = """
    COPY (
        SELECT recommendationid, appid, game, author_playtime_forever, review, votes_up, votes_funny, weighted_vote_score, received_for_free, written_during_early_access, voted_up
        FROM 'bronze/weighted_score_above_08.csv'
               WHERE language = 'english'
               AND NOT (recommendationid IS NULL 
                OR appid IS NULL
                OR game IS NULL
                OR review IS NULL
                OR votes_up IS NULL
                OR votes_funny IS NULL
                OR weighted_vote_score IS NULL
                OR received_for_free IS NULL
                OR author_playtime_forever IS NULL
                OR written_during_early_access IS NULL
                OR voted_up IS NULL
               )

    ) TO 'silver/cleaned_reviews.csv' (HEADER, DELIMITER ',');
"""

duckdb.sql(query)

In [42]:
query = """
    COPY (
        SELECT recommendationid, appid, game, author_playtime_forever, review, votes_up, votes_funny, weighted_vote_score, received_for_free, written_during_early_access, voted_up
        FROM 'bronze/weighted_score_above_08.csv'
               WHERE language = 'english'
               AND (recommendationid IS NULL 
                OR appid IS NULL
                OR game IS NULL
                OR review IS NULL
                OR votes_up IS NULL
                OR votes_funny IS NULL
                OR weighted_vote_score IS NULL
                OR received_for_free IS NULL
                OR author_playtime_forever IS NULL
                OR written_during_early_access IS NULL
                OR voted_up IS NULL
               )

    ) TO 'silver/malformed_reviews.csv' (HEADER, DELIMITER ',');
"""

duckdb.sql(query)

Helldivers analysis

In [52]:
query = """
       SELECT appid, name  
       FROM 'silver/cleaned_extended_steam_game_price_tiers.csv'
       WHERE name ILIKE '%zomboid%'
       LIMIT 5;

"""

duckdb.sql(query).show()

┌────────┬─────────────────┐
│ appid  │      name       │
│ int64  │     varchar     │
├────────┼─────────────────┤
│ 108600 │ Project Zomboid │
└────────┴─────────────────┘



In [53]:
query = """
    COPY (
        SELECT *
        FROM 'silver/cleaned_reviews.csv'
        WHERE appid = 108600

    ) TO 'silver/zomboid_reviews.csv' (HEADER, DELIMITER ',');
"""

duckdb.sql(query)

Analysis

In [3]:
def create_documents(df):
  documents = df['review'].tolist()
  return documents

In [ ]:
# optionally load model

loaded_model = BERTopic.load("models/countvectorized_model")

In [17]:
df = pd.read_csv('silver/zomboid_reviews.csv')

#analyzer = SentimentIntensityAnalyzer()
#df["vader_score"] = df["review"].apply(lambda x: analyzer.polarity_scores(x))

documents = create_documents(df)

umap_model = UMAP(n_neighbors=5, n_components=15,
                  min_dist=0.0, metric='cosine', random_state=42)

hdbscan_model = HDBSCAN(
    min_cluster_size=3, 
    min_samples=1, 

)

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True, bm25_weighting=True)

vectorizer_model = CountVectorizer(stop_words="english", min_df=2, max_features=10000)
topic_model = BERTopic(
                        vectorizer_model=vectorizer_model,
                        hdbscan_model=hdbscan_model,
                        ctfidf_model=ctfidf_model
                        )

# 3. Fit the model on your documents and predict topics
topics, probabilities = topic_model.fit_transform(documents)

# 4. View an overview table of all generated topics
print("--- Topic Overview ---")
print(topic_model.get_topic_info())

# 5. Extract the specific keywords for a target topic (e.g., Topic 0)
print("\n--- Keywords for Topic 0 ---")
print(topic_model.get_topic(0))

# Get the overview dataframe
df_topics = topic_model.get_topic_info()

# Save to a CSV file
df_topics.to_csv("silver/topic.csv", index=False)

# Convert vocabulary values from numpy types to native Python integers
if hasattr(topic_model.vectorizer_model, "vocabulary_"):
    topic_model.vectorizer_model.vocabulary_ = {
        k: int(v) for k, v in topic_model.vectorizer_model.vocabulary_.items()
    }

embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
topic_model.save("models/countvectorized_umap_reducedim", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)




Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3711.42it/s]


--- Topic Overview ---
    Topic  Count                                   Name  \
0      -1     29           -1_bigger_weapon_bullets_pen   
1       0     45         0_project_zomboid_step_certain   
2       1     12           1_kid_month_years_unfinished   
3       2     11                  2_friend_room_hes_ran   
4       3     11                  3_sex_beer_likes_boys   
5       4     11        4_options_tweaks_planned_design   
6       5     10            5_tasks_resident_lack_shove   
7       6     10           6_spare_decent_paint_average   
8       7      9                     7_1010_ate_died_2s   
9       8      9         8_thousand_stop_addicted_buddy   
10      9      8      9_market_version_simulator_cookie   
11     10      8             10_sims_dlc_neighbours_sim   
12     11      7           11_everyday_dawn_turn_sprint   
13     12      6  12_addictive_broadcasts_theyve_little   
14     13      6  13_tried_buddies_recovered_succumbing   
15     14      6         14_colle

In [ ]:
topic_model.reduce_topics(
    documents,
    nr_topics=30
)

topic_info = topic_model.get_topic_info()
print(topic_info.head(40))

# Save to a CSV file
df_topics.to_csv("silver/topic_overview_cv_30.csv", index=False)

   Topic  Count                        Name  \
0      0    273  0_game_zombie_zombies_just   
1      1     16    1_good_grind_bad_average   

                                      Representation  \
0  [game, zombie, zombies, just, like, time, zomb...   
1  [good, grind, bad, average, bugs, long, just, ...   

                                 Representative_Docs  
0  [Reviewing this game feels like a daunting tas...  
1  [---{ Graphics }--- ☐ You forget what reality ...  
